In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np

/opt/anaconda3/envs/ml_sys/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch, transformers, accelerate
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("accelerate", accelerate.__version__)

torch 2.7.0
transformers 5.1.0
accelerate 1.12.0


In [3]:
ds = load_dataset("Tobi-Bueck/customer-support-tickets")

In [7]:
train_ds = ds["train"]

In [35]:
ds["train"][0]

{'subject': 'Wesentlicher Sicherheitsvorfall',
 'body': 'Sehr geehrtes Support-Team,\\n\\nich möchte einen gravierenden Sicherheitsvorfall melden, der gegenwärtig mehrere Komponenten unserer Infrastruktur betrifft. Betroffene Geräte umfassen Projektoren, Bildschirme und Speicherlösungen auf Cloud-Plattformen. Der Grund für die Annahme ist, dass der Vorfall eine potenzielle Datenverletzung im Zusammenhang mit einer Cyberattacke darstellt, was ein erhebliches Risiko für sensible Informationen und den laufenden Geschäftsbetrieb unserer Organisation bedeutet.\\n\\nUnsere initialen Untersuchungen haben ungewöhnliche Aktivitäten und Abweichungen bei den Geräten ergeben. Trotz der Umsetzung unserer standardisierten Behebungs- und Eindämmungsmaßnahmen konnte die Bedrohung bislang nicht vollständig eliminiert.',
 'answer': 'Vielen Dank für die Meldung des kritischen Sicherheitsvorfalls und die Bereitstellung der Übersicht über die betroffenen Geräte sowie der ergriffenen ersten Maßnahmen. Wir e

In [5]:
ds["train"].column_names

['subject',
 'body',
 'answer',
 'type',
 'queue',
 'priority',
 'language',
 'version',
 'tag_1',
 'tag_2',
 'tag_3',
 'tag_4',
 'tag_5',
 'tag_6',
 'tag_7',
 'tag_8']

In [21]:
train_de_ds = train_ds.filter(lambda x: x["language"] == "de")

In [22]:
df = train_de_ds.to_pandas()

In [14]:
df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Anfrage nach detaillierten Angaben zur Systema...,"Sehr geehrtes Kundensupport-Team,\n\nich hoffe...",Vielen Dank für Ihre Anfrage. Wir stellen Ihne...,Request,Technical Support,low,de,51.0,Documentation,Feedback,IT,Tech Support,NaN,NaN,NaN,NaN
2,Anfrage zur Klärung der Auswirkungen eines Ser...,"Sehr geehrtes Kundendienstteam,\n\nich hoffe, ...","Vielen Dank, dass Sie uns bezüglich des kürzli...",Request,Service Outages and Maintenance,high,de,51.0,Disruption,Outage,Recovery,Support,NaN,NaN,NaN,NaN
3,Issue with SaaS Platform Functionality,"Sehr geehrtes Support-Team,\n\nich möchte Sie ...",Vielen Dank für Ihre Kontaktaufnahme bezüglich...,Incident,Product Support,medium,de,51.0,Bug,Performance,Disruption,Feature,NaN,NaN,NaN,NaN
4,Verbindungsstörung,"Sehr geehrtes Support-Team,\n\nich möchte ein ...","Vielen Dank, dass Sie sich wegen der Verbindun...",Problem,IT Support,high,de,51.0,Network,Disruption,Connectivity,Tech Support,NaN,NaN,NaN,NaN


In [15]:
df.shape

(33504, 16)

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 33504 entries, 0 to 33503
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   subject   31844 non-null  str    
 1   body      33503 non-null  str    
 2   answer    20321 non-null  str    
 3   type      20326 non-null  str    
 4   queue     33504 non-null  str    
 5   priority  33504 non-null  str    
 6   language  33504 non-null  str    
 7   version   12249 non-null  float64
 8   tag_1     20326 non-null  str    
 9   tag_2     20283 non-null  str    
 10  tag_3     20208 non-null  str    
 11  tag_4     18313 non-null  str    
 12  tag_5     11335 non-null  str    
 13  tag_6     5317 non-null   str    
 14  tag_7     2269 non-null   str    
 15  tag_8     864 non-null    str    
dtypes: float64(1), str(15)
memory usage: 31.0 MB


In [ ]:
import requests

BASE = "http://127.0.0.1:5001"

def lt_translate(text, source="de", target="en"):
    r = requests.post(
        f"{BASE}/translate",
        json={"q": text, "source": source, "target": target, "format": "text"},
        timeout=60
    )
    r.raise_for_status()
    return r.json()["translatedText"]

row = df.iloc[0]

subject_en = lt_translate(row["subject"])
body_en    = lt_translate(row["body"])
answer_en  = lt_translate(row["answer"]) if isinstance(row["answer"], str) else None

subject_en, body_en[:200], answer_en

('Essential safety incident',
 'Dear support team,\\n\\nich would like to report a serious safety incident that currently affects several components of our infrastructure. Affected devices include projectors, screens and storage solut',
 'Thank you for reporting the critical safety incident and providing the overview of the devices concerned and the first measures taken. We recognise the urgency and severity of the situation, and we are all committed to addressing the case priority. For an immediate investigation we need additional information: Please send us specific logs of projectors, screens and cloud storage systems, including time stamps of suspicious activities and unusual error messages. If possible, add a summary of the measures already taken.')

In [38]:
import requests

BASE = "http://127.0.0.1:5001"

def lt_translate(text, source="de", target="en"):
    if text is None:
        return None
    text = str(text)
    r = requests.post(
        f"{BASE}/translate",
        json={"q": text, "source": source, "target": target, "format": "text"},
        timeout=60
    )
    r.raise_for_status()
    return r.json()["translatedText"]

row = df.iloc[0]  # pick any row index you want

result = {
    "subject": lt_translate(row.get("subject")),
    "body": lt_translate(row.get("body")),
    "answer": lt_translate(row.get("answer")) if isinstance(row.get("answer"), str) else None,
}

result


{'subject': 'Essential safety incident',
 'body': 'Dear support team,\\n\\nich would like to report a serious safety incident that currently affects several components of our infrastructure. Affected devices include projectors, screens and storage solutions on cloud platforms. The reason for the assumption is that the incident is a potential breach of data related to a cyberattack, which means a significant risk for sensitive information and the ongoing operation of our organization. \\n\\nOur initial studies have shown unusual activities and deviations in the devices. Despite the implementation of our standardised recovery and containment measures, the threat has not yet been completely eliminated.',
 'answer': 'Thank you for reporting the critical safety incident and providing the overview of the devices concerned and the first measures taken. We recognise the urgency and severity of the situation, and we are all committed to addressing the case priority. For an immediate investigati

In [ ]:
import pandas as pd
import numpy as np
import requests

BASE = "http://127.0.0.1:5001"

def lt_translate(text, source="de", target="en"):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return None
    r = requests.post(
        f"{BASE}/translate",
        json={"q": str(text), "source": source, "target": target, "format": "text"},
        timeout=60
    )
    r.raise_for_status()
    return r.json()["translatedText"]

def clean_value(v):
    if pd.isna(v):
        return None
    if isinstance(v, (float, np.floating)) and float(v).is_integer():
        return int(v)
    return v

def make_english_ticket(row):
    row = row.to_dict() if hasattr(row, "to_dict") else dict(row)

    subject_en = lt_translate(row.get("subject"))
    body_en    = lt_translate(row.get("body"))
    answer_en  = lt_translate(row.get("answer"))

    out = {k: clean_value(v) for k, v in row.items()}
    out["subject"] = subject_en
    out["body"]    = body_en
    out["answer"]  = answer_en
    out["language"] = "en"
    return out

ticket_en = make_english_ticket(df.iloc[0])
ticket_en

{'subject': 'Essential safety incident',
 'body': 'Dear support team,\\n\\nich would like to report a serious safety incident that currently affects several components of our infrastructure. Affected devices include projectors, screens and storage solutions on cloud platforms. The reason for the assumption is that the incident is a potential breach of data related to a cyberattack, which means a significant risk for sensitive information and the ongoing operation of our organization. \\n\\nOur initial studies have shown unusual activities and deviations in the devices. Despite the implementation of our standardised recovery and containment measures, the threat has not yet been completely eliminated.',
 'answer': 'Thank you for reporting the critical safety incident and providing the overview of the devices concerned and the first measures taken. We recognise the urgency and severity of the situation, and we are all committed to addressing the case priority. For an immediate investigati